In [1]:
import urllib.request

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x108c20850>)

In [2]:
# load the veredict
import re

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

preprocessed_text = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed_text = [item.strip() for item in preprocessed_text if item.strip()]

In [3]:
all_words = sorted(set(preprocessed_text))
vocab = {word: i for i, word in enumerate(all_words)}
print("This are the first 10 words of the vocabulary:")
for i in range(10):
    print(f"{i}: {all_words[i]}")

This are the first 10 words of the vocabulary:
0: !
1: "
2: '
3: (
4: )
5: ,
6: --
7: .
8: :
9: ;


In [4]:
# Exercice:
# Implement a tokenizer class with arguments vocab
# The class needs to include two functions: encode and decode


class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {value: key for key, value in vocab.items()}

    def encode(self, text):

        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        return [self.str_to_int[word] for word in tokens]

    def decode(self, tokens):

        text = " ".join([self.int_to_str[word] for word in tokens])
        # remove spaces before the special characters
        text = re.sub(r'\s+([,.:;?_!"()\'])', r"\1", text)
        return text


tokenizer = SimpleTokenizerV1(vocab)
text = """It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))


[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [5]:
# Extend the vocabulary with two extra tokens:

all_words.extend(["<|unk|>", "<|endoftext|>"])
vocab = {word: i for i, word in enumerate(all_words)}


# Now we readapt the tokenizer
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {value: key for key, value in vocab.items()}

    def encode(self, text):

        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        tokens = [item if item in self.str_to_int else "<|unk|>" for item in tokens]
        return [self.str_to_int[word] for word in tokens]

    def decode(self, tokens):

        text = " ".join([self.int_to_str[word] for word in tokens])
        # remove spaces before the special characters
        text = re.sub(r'\s+([,.:;?_!"()\'])', r"\1", text)
        return text


text1 = "Hello. do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = text1 + " <|endoftext|> " + text2

tokenizer = SimpleTokenizerV2(vocab)
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))


[1130, 7, 355, 1126, 628, 975, 10, 1131, 55, 988, 956, 984, 722, 988, 1130, 7]
<|unk|>. do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [6]:
vocab["<|endoftext|>"]

1131

#  Byte Pair Encoding

In [7]:
# %pip install tiktoken

In [11]:
import tiktoken

In [12]:
tokenizer = tiktoken.get_encoding("gpt2")

In [13]:
sample_text = "akwirw ier"
ids = tokenizer.encode(sample_text)
print(ids)

decoded_tokens = tokenizer.decode(ids)
print(decoded_tokens)
assert decoded_tokens == sample_text

[461, 86, 343, 86, 220, 959]
akwirw ier


# Create dataset for training an LLM

In [14]:
tokenizer = tiktoken.get_encoding("gpt2")
with open("the-verdict.txt", "r") as f:
    raw_text = f.read()

encoded_text = tokenizer.encode(raw_text)
enc_sample = encoded_text[50:]  # The book says this makes it a bit more interesting.


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.outut_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            output_chunk = token_ids[i + 1 : i + 1 + max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.outut_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.outut_ids[idx]


def create_dataloader_v1(
    text,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader


In [16]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [17]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
[tensor([[2885, 1464]]), tensor([[1464, 1807]])]


In [18]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=2, shuffle=False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
[tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


In [19]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=2, max_length=8, stride=2, shuffle=False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899],
        [ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]
[tensor([[ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]]), tensor([[ 3619,   402,   271, 10899,  2138,   257,  7026, 15632],
        [  271, 10899,  2138,   257,  7026, 15632,   438,  2016]])]


# Words Embeddings

In [20]:
import torch

In [ ]:
vacb_size = 50257
output_dim = 256
token_embedding = torch.nn.Embedding(vacb_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Tokens = ", inputs)
print("Targets = ", targets)

Tokens =  tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets =  tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [22]:
token_embeddings = token_embedding(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [23]:
# You someteimes also define a position embedding
context_length = max_length
pos_embeddings_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embeddings_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [ ]:
# THe input embeddings will be:

input_embeddings = (
    token_embeddings + pos_embeddings
)  # It is important than while the shape is different the sum is added ot each of the elements.
print(input_embeddings.shape)

torch.Size([8, 4, 256])
